## How to run in Colab
1. Open [Google Colab](https://colab.research.google.com) → **File → Upload notebook** → choose this file.
2. Download the datasets first and save them in `C:\Users\sivan\Datasets`:
   - Exercises 1 to 4: click [titanic dataset.zip](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%204%20-%20Data%20Understanding/W4D4%20-%20Data%20Preprocessing%20&%20T/titanic%20dataset.zip). It downloads `titanic dataset.zip` (you already have it from Exercises XP).
   - Exercise 5: click [Superstore Sales Data.zip](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%204%20-%20Data%20Understanding/W4D4%20-%20Data%20Preprocessing%20&%20T/Superstore%20Sales%20Data.zip). It downloads `Superstore Sales Data.zip`.
   - Exercise 6: click [Air Quality Data in India.zip](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%204%20-%20Data%20Understanding/W4D4%20-%20Data%20Preprocessing%20&%20T/Air%20Quality%20Data%20in%20India.zip). It downloads `Air Quality Data in India.zip` (77 MB, so the upload to Colab takes a few minutes).
3. In each exercise, the upload cell asks for its file: choose it from `C:\Users\sivan\Datasets`:
   - Exercise 1: `titanic dataset.zip` (Exercises 2, 3 and 4 use the same file, so you upload it only once)
   - Exercise 5: `Superstore Sales Data.zip`
   - Exercise 6: `Air Quality Data in India.zip`

**Run the cells from top to bottom** (Shift+Enter runs one cell). After a restart, run them again from the top.

# What you will learn

- Perform scaling and normalization on numerical features in the Titanic dataset.
- Create new composite features in the Titanic dataset to enhance the data analysis.
- Apply Min-Max normalization to the ‘Sales’ and ‘Profit’ columns in an E-Commerce dataset.
- Aggregate daily air quality measurements to monthly averages.

# What you will create

- A version of the Titanic dataset with ‘Fare’ and ‘Age’ columns scaled and normalized.
- A transformed Titanic dataset featuring new composite attributes like ‘Family Size’ and ‘IsAlone’.
- A version of the Titanic dataset with ‘Age’ and ‘Fare’ columns normalized using Min-Max and Z-score normalization.
- A Superstore Sales dataset with ‘Sales’ and ‘Profit’ columns normalized using Min-Max normalization.
- The Air Quality dataset grouped by location and month, with average key measurements calculated for each month.

## Setup
Run this cell first. It imports the libraries used in all the exercises.

In [ ]:
import pandas as pd
# pandas: for working with tables (DataFrames)
import matplotlib.pyplot as plt
# matplotlib: for drawing charts
import seaborn as sns
# seaborn: for nicer statistical charts (built on matplotlib)
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
# StandardScaler = Z-score scaling, MinMaxScaler = Min-Max normalization, LabelEncoder turns text categories into numbers

---
# Exercise 1: Data Scaling and Normalization

**Instructions**

- Identify numerical columns in the dataset like ‘Fare’ and ‘Age’.
- Apply standardization (Z-score scaling) to features with a Gaussian distribution.
- Apply Min-Max normalization to features that require bounded ranges.
- Analyze the effect of scaling and normalization on model performance.

*Hint: Use StandardScaler and MinMaxScaler from scikit-learn.*

### Step 1: Load the Titanic data and identify the numerical columns

In [ ]:
from google.colab import files
# the Colab tool for uploading files from your computer
uploaded = files.upload()
# choose "titanic dataset.zip" (saved in a variable so its contents are not printed)
!unzip "titanic dataset.zip"
# unzip it: you get the folder "titanic dataset" with train.csv, test.csv and gender_submission.csv

In [ ]:
titanic = pd.read_csv('titanic dataset/train.csv')
# load train.csv (it has the Survived column we use to test a model)
print(titanic.shape)
# rows and columns
print(titanic.select_dtypes(include='number').columns.tolist())
# the numerical columns (int or float)
print(titanic[['Age', 'Fare']].isnull().sum())
# missing values in Age and Fare

**Explanation:** The table has **891 rows and 12 columns**. The numerical columns are PassengerId, Survived, Pclass, Age, SibSp, Parch and Fare. PassengerId is only a row number and Survived is the answer we predict, so the real numerical **features** are **Pclass, Age, SibSp, Parch and Fare**. **Age has 177 missing values** and Fare has none, so we fill Age with the median (28) in the next cell: the model in Step 4 cannot use empty cells.

In [ ]:
titanic['Age'] = titanic['Age'].fillna(titanic['Age'].median())
# fill the 177 missing ages with the median age (the scalers and the model need a value in every row)
print('Missing Age now:', titanic['Age'].isnull().sum())
# check: 0

### Step 2: Which columns look Gaussian (bell-shaped)?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# one row of 2 charts
sns.histplot(titanic['Age'], bins=30, ax=axes[0])
# histogram of Age
axes[0].set_title('Age')
# chart title
sns.histplot(titanic['Fare'], bins=30, ax=axes[1])
# histogram of Fare
axes[1].set_title('Fare')
# chart title
plt.tight_layout()
# leave space between the charts
plt.show()
# show the charts

In [ ]:
print(titanic[['Age', 'Fare', 'SibSp', 'Parch', 'Pclass']].skew().round(2))
# skew: near 0 = symmetric like a bell, above 1 = a long tail on one side

**Choice:** The histogram of **Age** looks like a bell (one hump in the middle) and its skew is only **0.51**, so Age is close to **Gaussian** → **standardization (Z-score)**. **Fare** has a very long tail to the right: most fares are small, a few are huge, and its skew is **4.79** → not Gaussian, so it gets **Min-Max normalization** (a bounded 0 to 1 range). SibSp (**3.70**) and Parch (**2.75**) are skewed too, and Pclass only takes the values 1, 2 and 3 (bounded), so in the model test they also get Min-Max.

### Step 3: Standardize Age and normalize Fare

In [ ]:
z_scaler = StandardScaler()
# create the Z-score scaler (mean 0, std 1)
titanic['Age_standardized'] = z_scaler.fit_transform(titanic[['Age']])
# (age - mean) / std, saved in a new column
minmax_scaler = MinMaxScaler()
# create the Min-Max scaler (range 0 to 1)
titanic['Fare_normalized'] = minmax_scaler.fit_transform(titanic[['Fare']])
# (fare - min) / (max - min), saved in a new column
print(titanic[['Age', 'Age_standardized', 'Fare', 'Fare_normalized']].describe().round(2))
# the original and the new columns side by side

**Explanation:** `Age_standardized` has a **mean of 0.00 and a std of 1.00** (from 29.36 and 13.02): the ages now go from **-2.22 to 3.89**, the number of standard deviations from the average. `Fare_normalized` goes from **0.00 to 1.00** (from 0 to 512.33). This is the version of the dataset "with Fare and Age scaled and normalized". Its median is only **0.03** and its 75% value **0.06**: Min-Max does not change the shape, so most fares are still squeezed near 0 because of the few very big fares.

### Step 4: The effect of scaling and normalization on model performance
We train the same model 3 times on `Survived`, and change only the scaling:
- **A:** no scaling
- **B:** our choice from Step 2: Age with StandardScaler, Fare, Pclass, SibSp and Parch with MinMaxScaler
- **C:** StandardScaler on every feature

The model is **K-Nearest Neighbors (KNN)**: it predicts a passenger from the 5 most similar passengers, and "similar" is a **distance** between the numbers, so it is sensitive to scale.

In [ ]:
encoder = LabelEncoder()
# create the label encoder
titanic['Sex_encoded'] = encoder.fit_transform(titanic['Sex'])
# female -> 0, male -> 1 (the model needs numbers)
features = ['Pclass', 'Sex_encoded', 'Age', 'SibSp', 'Parch', 'Fare']
# the information the model uses
X = titanic[features]
# the feature table
y = titanic['Survived']
# the target: 1 = survived, 0 = did not
from sklearn.model_selection import train_test_split
# the tool that splits the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# 80% of the rows to train, 20% to test; random_state=42 gives the same split every time
print('Train rows:', len(X_train), '| Test rows:', len(X_test))
# the size of each part

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
# the KNN model
from sklearn.metrics import accuracy_score
# accuracy = the share of correct predictions
model_a = KNeighborsClassifier(n_neighbors=5)
# version A: a KNN model that looks at the 5 nearest passengers
model_a.fit(X_train, y_train)
# train it on the original numbers
accuracy_a = accuracy_score(y_test, model_a.predict(X_test))
# predict the test passengers and compare with the real answers
print('A - no scaling:', round(accuracy_a, 3))
# show the accuracy

In [ ]:
X_train_b = X_train.copy()
# a copy of the training features for version B
X_test_b = X_test.copy()
# a copy of the test features for version B
age_scaler = StandardScaler()
# Z-score scaler for Age
X_train_b[['Age']] = age_scaler.fit_transform(X_train[['Age']])
# learn the mean and std of Age from the training rows only, and scale them
X_test_b[['Age']] = age_scaler.transform(X_test[['Age']])
# scale the test ages with the same mean and std
minmax_cols = ['Pclass', 'SibSp', 'Parch', 'Fare']
# the skewed or bounded columns
range_scaler = MinMaxScaler()
# Min-Max scaler for those columns
X_train_b[minmax_cols] = range_scaler.fit_transform(X_train[minmax_cols])
# learn the min and max from the training rows only, and scale them
X_test_b[minmax_cols] = range_scaler.transform(X_test[minmax_cols])
# scale the test rows with the same min and max
model_b = KNeighborsClassifier(n_neighbors=5)
# version B: the same model
model_b.fit(X_train_b, y_train)
# train it on the scaled and normalized features
accuracy_b = accuracy_score(y_test, model_b.predict(X_test_b))
# accuracy on the test rows
print('B - Age standardized + others Min-Max:', round(accuracy_b, 3))
# show the accuracy

In [ ]:
all_scaler = StandardScaler()
# version C: one Z-score scaler for all features
X_train_c = all_scaler.fit_transform(X_train)
# learn from the training rows and scale them
X_test_c = all_scaler.transform(X_test)
# scale the test rows the same way
model_c = KNeighborsClassifier(n_neighbors=5)
# the same model
model_c.fit(X_train_c, y_train)
# train it
accuracy_c = accuracy_score(y_test, model_c.predict(X_test_c))
# accuracy on the test rows
print('C - StandardScaler on all:', round(accuracy_c, 3))
# show the accuracy

In [ ]:
results = pd.Series({'A: no scaling': accuracy_a, 'B: Z-score + Min-Max': accuracy_b, 'C: Z-score on all': accuracy_c})
# put the three accuracies together
results.plot(kind='bar', color=['gray', 'green', 'steelblue'])
# bar chart of the three accuracies
plt.ylabel('Accuracy')
# y axis label
plt.ylim(0, 1)
# y axis from 0 to 1
plt.xticks(rotation=0)
# keep the labels horizontal
plt.title('KNN accuracy on Survived')
# chart title
plt.show()
# show the chart

**Explanation (effect on model performance):** With the same KNN model and the same 712 training / 179 test passengers:
- **A, no scaling: 0.687** accuracy. KNN measures distances, and Fare (0 to 512) is so much bigger than Sex (0 or 1) or Pclass (1 to 3) that it almost decides the "nearest" passengers alone.
- **B, Age standardized + the others Min-Max: 0.816**, the best result (+12.9 points).
- **C, StandardScaler on everything: 0.788**, also much better than A.

So scaling clearly improves this model, and our choice (Z-score for the bell-shaped Age, Min-Max for the skewed or bounded columns) gave the best score here. The scalers learned their numbers from the training rows only, so the test rows stay unseen. This is one split of 179 passengers, so the small gap between B and C (0.028, 5 passengers) is less certain than the big gap with A.

---
# Exercise 2: Creating Composite Features

**Instructions**

- Create a new feature, ‘Family Size’, by combining ‘SibSp’ (siblings and spouses) and ‘Parch’ (parents and children).
- Create a ‘IsAlone’ feature to indicate whether a passenger is traveling alone.
- Explore the relationship between these new features and the survival rate.

*Hint: Use basic arithmetic operations and conditional statements in Pandas.*

### Step 1: Create 'Family Size' and 'IsAlone'

In [ ]:
titanic['Family Size'] = titanic['SibSp'] + titanic['Parch'] + 1
# siblings/spouses + parents/children + 1 for the passenger
titanic['IsAlone'] = 0
# start with 0 (not alone) for everyone
titanic.loc[titanic['Family Size'] == 1, 'IsAlone'] = 1
# condition: if the family size is 1, the passenger travels alone -> 1
print(titanic[['SibSp', 'Parch', 'Family Size', 'IsAlone']].head())
# check the first rows
print(titanic['IsAlone'].value_counts())
# how many passengers travel alone (1) and with family (0)

### Step 2: The relationship with the survival rate

In [ ]:
alone_survival = titanic.groupby('IsAlone')['Survived'].agg(['mean', 'count'])
# for alone (1) and not alone (0): the survival rate and the number of passengers
print(alone_survival.round(2))
# show the table

In [ ]:
family_survival = titanic.groupby('Family Size')['Survived'].agg(['mean', 'count'])
# for each family size: the survival rate and the number of passengers
print(family_survival.round(2))
# show the table

In [ ]:
print(titanic[['Family Size', 'IsAlone', 'Survived']].corr().round(2))
# correlation of the new features with Survived (near 0 = weak, close to 1 or -1 = strong)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# one row of 2 charts
alone_survival['mean'].plot(kind='bar', ax=axes[0], color=['green', 'gray'])
# survival rate: not alone (0) and alone (1)
axes[0].set_title('Survival rate by IsAlone')
# chart title
axes[0].set_xticks([0, 1], ['with family (0)', 'alone (1)'], rotation=0)
# clearer labels
family_survival['mean'].plot(kind='bar', ax=axes[1], color='steelblue')
# survival rate for each family size
axes[1].set_title('Survival rate by Family Size')
# chart title
plt.tight_layout()
# leave space between the charts
plt.show()
# show the charts

**Explanation:**
- **Family Size** = SibSp + Parch + 1. Row 0 has 1 sibling/spouse and 0 parents/children → family size **2**. **IsAlone** is 1 when the family size is 1: **537** passengers travel alone and **354** with family.
- **IsAlone vs survival:** **51%** of the passengers with family survived, but only **30%** of those alone.
- **Family Size vs survival:** it is not a straight line. Survival goes up from **30%** (size 1) to **55%** (2), **58%** (3) and **72%** (4), then drops for big families: **20%** (5), **14%** (6), and **0%** for sizes 8 and 11 (6 and 7 passengers). Small families did best.
- That is why the correlation of **Family Size with Survived is only 0.02**: the rise and the fall cancel out in a single straight-line number. **IsAlone** shows the link better (**-0.20**: alone → lower survival).
- Note: the groups for sizes 5 to 11 are small (6 to 22 passengers), so their exact rates are less reliable.

---
# Exercise 3: Data Normalization on the Titanic Dataset

**Instructions**

- Import the titanic dataset using Pandas in Python.
- Apply Min-Max normalization and Z-score normalization to these columns : Age and Fare
- Compare the distributions before and after normalization using histograms.

### Step 1: Import the Titanic dataset with pandas

In [ ]:
titanic_raw = pd.read_csv('titanic dataset/train.csv')
# import the file again, so we start from the original values (already unzipped in Exercise 1)
print(titanic_raw[['Age', 'Fare']].describe().round(2))
# the original Age and Fare
print('Missing Age:', titanic_raw['Age'].isnull().sum())
# empty ages

We do **not** fill the 177 missing ages here: the exercise is about the distribution of the real values, and the scikit-learn scalers skip empty cells and leave them empty (the check below proves it).

### Step 2: Min-Max normalization and Z-score normalization of Age and Fare

In [ ]:
minmax = MinMaxScaler()
# Min-Max: (value - min) / (max - min) -> range 0 to 1
titanic_raw[['Age_minmax', 'Fare_minmax']] = minmax.fit_transform(titanic_raw[['Age', 'Fare']])
# normalize both columns and save them as new columns
zscore = StandardScaler()
# Z-score: (value - mean) / std -> mean 0, std 1
titanic_raw[['Age_zscore', 'Fare_zscore']] = zscore.fit_transform(titanic_raw[['Age', 'Fare']])
# standardize both columns and save them as new columns
print(titanic_raw[['Age_minmax', 'Age_zscore', 'Fare_minmax', 'Fare_zscore']].describe().round(2))
# the new columns
print('Missing Age_minmax:', titanic_raw['Age_minmax'].isnull().sum(), '| Missing Age_zscore:', titanic_raw['Age_zscore'].isnull().sum())
# the 177 empty ages stay empty

In [ ]:
print(titanic_raw[['Age', 'Age_minmax', 'Age_zscore', 'Fare', 'Fare_minmax', 'Fare_zscore']].skew().round(2))
# the skew (the shape of the distribution) of each version

### Step 3: Compare the distributions with histograms

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
# 2 rows (Age, Fare) x 3 columns (original, Min-Max, Z-score)
sns.histplot(titanic_raw['Age'], bins=30, ax=axes[0, 0])
# Age original
axes[0, 0].set_title('Age - original')
# chart title
sns.histplot(titanic_raw['Age_minmax'], bins=30, ax=axes[0, 1], color='green')
# Age after Min-Max
axes[0, 1].set_title('Age - Min-Max (0 to 1)')
# chart title
sns.histplot(titanic_raw['Age_zscore'], bins=30, ax=axes[0, 2], color='orange')
# Age after Z-score
axes[0, 2].set_title('Age - Z-score (mean 0, std 1)')
# chart title
sns.histplot(titanic_raw['Fare'], bins=30, ax=axes[1, 0])
# Fare original
axes[1, 0].set_title('Fare - original')
# chart title
sns.histplot(titanic_raw['Fare_minmax'], bins=30, ax=axes[1, 1], color='green')
# Fare after Min-Max
axes[1, 1].set_title('Fare - Min-Max (0 to 1)')
# chart title
sns.histplot(titanic_raw['Fare_zscore'], bins=30, ax=axes[1, 2], color='orange')
# Fare after Z-score
axes[1, 2].set_title('Fare - Z-score (mean 0, std 1)')
# chart title
plt.tight_layout()
# leave space between the charts
plt.show()
# show the charts

**Explanation:**
- **Min-Max:** Age_minmax and Fare_minmax go from **0.00 to 1.00**. **Z-score:** Age_zscore and Fare_zscore have a **mean of 0.00 and a std of 1.00**. Age now goes from **-2.02 to 3.47** and Fare from **-0.65 to 9.67**. That 9.67 means the biggest fare is almost 10 standard deviations above the average, which is the far-right outlier.
- The **177** empty ages stayed empty in both new columns, so the scalers used only the 714 real ages.
- **The shape did not change:** the skew is **0.39** for all three Age versions and **4.79** for all three Fare versions. The histograms show the same picture: in each row, the 3 charts have the same bars, and only the numbers on the x-axis change. Normalization changes the **scale** (the units), not the **distribution**. The long tail of Fare stays, so if we want to reduce it we need an outlier treatment or a log transformation (as in Exercises XP), not a scaler.

---
# Exercise 4 : Data Reduction and Aggregation

**Instructions**

- Perform data reduction on the titanic dataset by implementing dimensionality reduction techniques like Principal Component Analysis (PCA).
- Aggregate the data by a categorical column (like date or region) and calculate summary statistics (mean, sum, etc.).
- Visualize the aggregated data using appropriate plots.

### Step 1: Data reduction with PCA

In [ ]:
numeric_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
# the numerical columns of the titanic table from Exercise 1 (Age is already filled there)
print(titanic[numeric_cols].isnull().sum())
# check: no missing values
scaled_numbers = StandardScaler().fit_transform(titanic[numeric_cols])
# scale them first, so Fare (big numbers) does not dominate PCA

In [ ]:
from sklearn.decomposition import PCA
# PCA reduces many columns to a few new ones
pca = PCA(n_components=2)
# keep 2 components
components = pca.fit_transform(scaled_numbers)
# 2 new values for every passenger instead of 5
print('Shape before:', scaled_numbers.shape, '| after:', components.shape)
# 5 columns -> 2 columns
print('Variance kept by PC1, PC2:', pca.explained_variance_ratio_.round(3))
# the share of the information each component keeps
print('Total:', round(pca.explained_variance_ratio_.sum(), 3))
# both together
print(pd.DataFrame(pca.components_.T, index=numeric_cols, columns=['PC1', 'PC2']).round(2))
# how much each column counts in each component

In [ ]:
titanic['PC1'] = components[:, 0]
# first component of each passenger
titanic['PC2'] = components[:, 1]
# second component
sns.scatterplot(data=titanic, x='PC1', y='PC2', hue='Pclass', palette='Set1', alpha=0.6)
# one point per passenger, colored by class
plt.title('Titanic passengers after PCA')
# chart title
plt.show()
# show the chart

### Step 2: Aggregate by a categorical column (Pclass and Embarked) with summary statistics

In [ ]:
by_class = titanic.groupby('Pclass').agg(
    passengers=('Survived', 'count'),
    survivors=('Survived', 'sum'),
    survival_rate=('Survived', 'mean'),
    mean_fare=('Fare', 'mean'),
    total_fare=('Fare', 'sum'),
    mean_age=('Age', 'mean'),
    mean_PC1=('PC1', 'mean')
)
# for each class: the number of passengers, survivors (sum of 1s), survival rate, mean and total fare, mean age and mean PC1
print(by_class.round(2))
# show the table

In [ ]:
by_port = titanic.groupby('Embarked').agg(
    passengers=('Survived', 'count'),
    survivors=('Survived', 'sum'),
    survival_rate=('Survived', 'mean'),
    mean_fare=('Fare', 'mean')
)
# the same summary for each port (C = Cherbourg, Q = Queenstown, S = Southampton)
print(by_port.round(2))
# show the table

### Step 3: Visualize the aggregated data

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
# one row of 3 charts
by_class['survival_rate'].plot(kind='bar', ax=axes[0], color='steelblue')
# survival rate by class
axes[0].set_title('Survival rate by Pclass')
# chart title
by_class['mean_fare'].plot(kind='bar', ax=axes[1], color='orange')
# average fare by class
axes[1].set_title('Average Fare by Pclass')
# chart title
by_port['survival_rate'].plot(kind='bar', ax=axes[2], color='green')
# survival rate by port
axes[2].set_title('Survival rate by Embarked')
# chart title
plt.tight_layout()
# leave space between the charts
plt.show()
# show the charts

**Explanation:**
- **Reduction:** PCA turned the 5 scaled numerical columns into **2 components** (shape **(891, 5) → (891, 2)**). The two keep **34.0% + 32.5% = 66.5%** of the information. **PC1** is mostly class and fare (Pclass **0.68**, Fare **-0.56**, Age **-0.46**): a high PC1 means 3rd class, cheap ticket, younger. **PC2** is mostly family (SibSp **0.62**, Parch **0.62**). In the scatter plot, the classes form separate bands along PC1.
- **Aggregation by Pclass:** 1st class had **216** passengers and **136** survivors (**63%**), 2nd class 87 of 184 (**47%**), and 3rd class only 119 of 491 (**24%**). The mean fare falls from **84.15** to **20.66** and **13.68**, and the mean age from **36.81** to **29.77** and **25.93**. The mean PC1 rises from **-1.92** to **-0.15** and **0.90**, which confirms that PC1 is the class axis.
- **Aggregation by Embarked:** Cherbourg (C) has the best survival, **55%** (93 of 168), and the highest mean fare, **59.95**. Queenstown (Q) has **39%** and Southampton (S) **34%** (217 of 644). The bar charts show both patterns: more expensive groups survived more.

---
# Exercise 5: Normalizing E-Commerce Sales Data

**Instructions**

Dataset: Use the [Superstore Sales Data](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%204%20-%20Data%20Understanding/W4D4%20-%20Data%20Preprocessing%20&%20T/Superstore%20Sales%20Data.zip).

- Load the Superstore Sales dataset.
- Perform Min-Max normalization on the ‘Sales’ and ‘Profit’ columns.
- Create new columns, e.g., ‘Sales_normalized’ and ‘Profit_normalized’, to store the normalized values.

### Step 1: Load the Superstore Sales dataset

In [ ]:
uploaded = files.upload()
# choose "Superstore Sales Data.zip"
!unzip "Superstore Sales Data.zip"
# unzip it: you get the folder "Superstore Sales Data" with "superstore_dataset2011-2015.csv"

In [ ]:
sales = pd.read_csv('Superstore Sales Data/superstore_dataset2011-2015.csv', encoding='latin1')
# load the file; encoding='latin1' is needed because some names have special letters (the normal utf-8 gives an error)
print(sales.shape)
# rows and columns
print(sales.columns.tolist())
# the real column names
print(sales[['Sales', 'Profit']].isnull().sum())
# missing values in the two columns
print(sales[['Sales', 'Profit']].describe().round(2))
# the original range of Sales and Profit

**Explanation:** The file has **51,290 orders and 24 columns**, and **Sales** and **Profit** have **no missing values**. The ranges are very different: Sales goes from **0.44 to 22,638.48**, and Profit from **-6,599.98** (a loss) to **8,399.98**. That is why we normalize them, to bring both to the same 0 to 1 scale.

### Step 2: Min-Max normalization into new columns

In [ ]:
sales_scaler = MinMaxScaler()
# create the Min-Max scaler
sales[['Sales_normalized', 'Profit_normalized']] = sales_scaler.fit_transform(sales[['Sales', 'Profit']])
# each column separately: (value - min) / (max - min), saved in the 2 new columns
print(sales[['Sales', 'Sales_normalized', 'Profit', 'Profit_normalized']].head())
# the original and normalized values side by side
print(sales[['Sales_normalized', 'Profit_normalized']].describe().round(3))
# the new columns: min 0 and max 1

In [ ]:
profit_min = sales['Profit'].min()
# the lowest profit (a loss)
profit_max = sales['Profit'].max()
# the highest profit
zero_profit_position = (0 - profit_min) / (profit_max - profit_min)
# where a profit of exactly 0 lands after Min-Max (the same formula the scaler uses)
print('A profit of 0 becomes:', round(zero_profit_position, 3))
# show it
print('Orders with a loss (Profit < 0):', (sales['Profit'] < 0).sum())
# how many orders lost money

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# one row of 2 charts
sns.histplot(sales['Profit'], bins=60, ax=axes[0])
# Profit before
axes[0].set_title('Profit - original')
# chart title
sns.histplot(sales['Profit_normalized'], bins=60, ax=axes[1], color='green')
# Profit after Min-Max
axes[1].set_title('Profit_normalized (0 to 1)')
# chart title
plt.tight_layout()
# leave space between the charts
plt.show()
# show the charts

**Explanation:**
- The two new columns **Sales_normalized** and **Profit_normalized** both go from **0.000 to 1.000** (describe min and max). The first row shows the formula at work: a sale of 408.30 became **0.018**.
- **Profit_normalized** has no negative numbers anymore: the biggest loss (-6,599.98) is 0, and a profit of exactly **0 is now 0.44**. So every value below 0.44 is a loss, and there are **12,544** loss orders.
- **Min-Max keeps the outliers:** the 25%, 50% and 75% values of Profit_normalized are all between **0.440 and 0.442**, and the std is only **0.012**. The few huge profits and losses take up the ends of the 0 to 1 range, and almost all normal orders are squeezed in the middle. It is the same for Sales: its mean is only **0.011**. The histograms show it: the same narrow spike before and after, and only the x-axis numbers change.

---
# Exercise 6: Aggregating Air Quality Data

**Instructions**

Dataset: Use the [Air Quality Data in India](https://github.com/devtlv/Datasets-DA-Bootcamp-2-/raw/refs/heads/main/Week%204%20-%20Data%20Understanding/W4D4%20-%20Data%20Preprocessing%20&%20T/Air%20Quality%20Data%20in%20India.zip).

- Load the Air Quality dataset.
- Convert the ‘Date’ column to a datetime format.
- Group the data by location and month, calculating the average of key measurements (e.g., PM2.5, PM10, NO2) for each month.
- Store the aggregated data in a new DataFrame and analyze trends in air quality over time.

### Step 1: Load the Air Quality dataset and check the column names

In [ ]:
uploaded = files.upload()
# choose "Air Quality Data in India.zip" (77 MB, the upload takes a few minutes)
!unzip "Air Quality Data in India.zip"
# unzip it: you get the folder "Air Quality Data in India" with 5 files; we use "city_day.csv" (daily values per city)

In [ ]:
air = pd.read_csv('Air Quality Data in India/city_day.csv')
# one row per city per day
print(air.shape)
# rows and columns
print(air.columns.tolist())
# the real column names
print(air[['City', 'Date', 'PM2.5', 'PM10', 'NO2']].isnull().sum())
# missing values in the columns we use

**Explanation:** `city_day.csv` has **29,531 rows** (one city on one day) and **16 columns**. The exercise says "location": in this file the location column is called **City**. **PM2.5, PM10 and NO2** exist, but have many empty values: **4,598**, **11,140** and **3,585**. We do not fill them, because inventing pollution values would change the averages. `mean()` simply skips the empty days.

### Step 2: Convert 'Date' to datetime

In [ ]:
print(air['Date'].dtype)
# before: text (object)
air['Date'] = pd.to_datetime(air['Date'])
# turn the text dates (year-month-day) into real dates
print(air['Date'].dtype)
# after: datetime64
print(air['Date'].min(), '->', air['Date'].max())
# the first and last day
print('Cities:', air['City'].nunique())
# how many cities

### Step 3: Group by location (City) and month, and store the averages in a new DataFrame

In [ ]:
month_period = air['Date'].dt.to_period('M')
# the year and month of each day, e.g. 2015-01
air['Month'] = month_period.dt.to_timestamp()
# turn it into the first day of that month (a real date, good for charts)
key_measurements = ['PM2.5', 'PM10', 'NO2']
# the key measurements the exercise names
monthly = air.groupby(['City', 'Month'])[key_measurements].mean()
# the average of each measurement for every city and month (empty days are skipped)
monthly = monthly.reset_index()
# turn City and Month back into normal columns
print(monthly.shape)
# rows = city-month pairs
print(monthly.head())
# the first rows of the new DataFrame
print(monthly[key_measurements].isnull().sum())
# city-months with no measurement at all for that pollutant

### Step 4: Analyze trends in air quality over time

In [ ]:
monthly['Month number'] = monthly['Month'].dt.month
# 1 = January ... 12 = December
season = monthly.groupby('Month number')[key_measurements].mean()
# the average of all city-months for each month of the year
print(season.round(1))
# show the seasonal pattern

In [ ]:
monthly['Year'] = monthly['Month'].dt.year
# the year of each row
print(monthly.groupby('Year')['City'].nunique())
# how many cities have data in each year
delhi = monthly[monthly['City'] == 'Delhi']
# only Delhi: one city with data in every year, so its trend is fair
delhi_by_year = delhi.groupby('Year')[key_measurements].mean()
# Delhi's yearly average of the monthly averages
print(delhi_by_year.round(1))
# show it

In [ ]:
city_pm25 = monthly.groupby('City')['PM2.5'].mean().sort_values(ascending=False)
# the average monthly PM2.5 of each city, from the most polluted to the least
print(city_pm25.round(1).head(5))
# the 5 most polluted cities
print(city_pm25.round(1).tail(5))
# the 5 least polluted cities

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
# one row of 3 charts
for col in key_measurements:
# one line per measurement
    axes[0].plot(delhi['Month'], delhi[col], label=col)
    # Delhi's monthly average over time
axes[0].legend()
# show which line is which
axes[0].set_title('Delhi - monthly averages over time')
# chart title
season.plot(kind='bar', ax=axes[1])
# the average by month of the year
axes[1].set_title('All cities - average by month of the year')
# chart title
city_pm25.head(10).plot(kind='bar', ax=axes[2], color='brown')
# the 10 most polluted cities
axes[2].set_title('Top 10 cities by average monthly PM2.5')
# chart title
plt.tight_layout()
# leave space between the charts
plt.show()
# show the charts

**Explanation:**
- **Datetime:** `Date` changed from **object** (text) to **datetime64**. The data runs from **2015-01-01 to 2020-07-01** in **26 cities**.
- **The new DataFrame `monthly`** has **1,005 rows**, one per city and month, with the average PM2.5, PM10 and NO2. Some city-months have no value at all for a pollutant (**112** for PM2.5, **325** for PM10, **71** for NO2). For example, Ahmedabad has no PM10 readings in its first months (NaN in the first 5 rows, January to May 2015).
- **Seasonal trend (strongest pattern):** pollution is highest in winter and lowest in the monsoon. PM2.5 averages **106.3** in January, **110.3** in November and **109.2** in December, but only **34.7** in July and **32.8** in August. PM10 and NO2 follow the same curve (PM10 **169.6** in December vs **67.1** in July). The Delhi line chart shows a peak every winter.
- **Trend over the years:** the number of cities grows from **9** in 2015 to **26** in 2020, so an all-city yearly average would mix different cities. That is why we look at **Delhi only**: PM2.5 is **117.7** (2015), peaks at **138.6** (2016), then goes down to **124.9**, **115.1** and **108.5**, and **76.7** in 2020. Two cautions about 2020: it only has January to July, without the polluted November and December, and it includes the COVID lockdown months, so its low value is not a fair full-year comparison.
- **Locations:** the most polluted cities by average monthly PM2.5 are **Patna 121.5**, **Gurugram 116.6**, **Delhi 116.3** and **Lucknow 109.2**. The cleanest is **Aizawl, 15.8**.